# CONDA toxicity detection — k=0 with **class-weighted** training

This notebook fine-tunes **DeBERTa-v3-base** on the CONDA dataset with **k=0 previous messages** (no context) and **class-weighted cross-entropy** loss.

It is the weighted counterpart to `conda_k0.ipynb`. Everything else — data, model, tokenizer, hyperparameters — is identical, so any change in metrics is attributable to the loss function.

**Why:** CONDA is imbalanced (O ≈ 74% of train, I ≈ 6%). The unweighted k=0 model gets class-I precision 0.92 but recall only 0.61. Running weighted CE at k=0 tells us how much of the implicit-toxicity gap is an imbalance problem vs. a context problem.

**Output:** a saved model directory on Google Drive (`k0_weighted_model`).

**Runtime:** ≈20 min on a Colab A100, longer on T4.

## 1. Setup — install deps, mount Drive

In [ ]:
# Install dependencies (Colab usually has torch + transformers; sentencepiece is needed for DeBERTa-v3)
!pip install -q sentencepiece protobuf

In [ ]:
from pathlib import Path

K = 0
# Sauvegarde locale (pas de Drive). Sur Colab, /content/ est le disque local de la session.
# Sur ta machine, mets juste './conda_ksweep' ou n'importe quel chemin.
SAVE_ROOT = Path('/content/conda_ksweep')   # ← remplace par './conda_ksweep' si tu tournes en local
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_DIR = SAVE_ROOT / f'k{K}_weighted_model'
CKPT_DIR  = SAVE_ROOT / f'k{K}_weighted_checkpoints'
MODEL_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

print(f'Will save final model to: {MODEL_DIR}')
print(f'Training checkpoints in : {CKPT_DIR}')

Will save final model to: /content/conda_ksweep/k0_weighted_model
Training checkpoints in : /content/conda_ksweep/k0_weighted_checkpoints


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
import json, time, gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert device.type == 'cuda', 'GPU not available! Runtime > Change runtime type > GPU'

Device: cuda


## 2. Load CONDA dataset

Upload the three CONDA CSV files: `CONDA_train.csv`, `CONDA_valid.csv`, `CONDA_test.csv` (Ctrl+click to select all three at once).

In [ ]:
from google.colab import files
uploaded = files.upload()

train_df = pd.read_csv('CONDA_train.csv')
valid_df = pd.read_csv('CONDA_valid.csv')
print(f'Train: {train_df.shape}, Valid: {valid_df.shape}')
train_df.head()

Saving CONDA_test.csv to CONDA_test.csv
Saving CONDA_train.csv to CONDA_train.csv
Saving CONDA_valid.csv to CONDA_valid.csv
Train: (26921, 10), Valid: (8974, 10)


,Id,matchId,conversationId,utterance,chatTime,playerSlot,playerId,intentClass,slotClasses,slotTokens
0,11263,697,3193,wow!,76,0,ANTS IN MY EYES JOHNSON,O,O,"wow (O),"
1,13741,843,3809,WTF,1563,5,M.k,O,T,"WTF (T),"
2,22125,1412,6199,wpe wpe,2853,1,Acqua Ragia,O,O O,"wpe (O), wpe (O),"
3,6453,439,1875,hahaha,1038,0,juicebox,O,O,"hahaha (O),"
4,9644,601,2713,wtf,1661,5,KAIST.Shadows,O,T,"wtf (T),"


## 3. Preprocessing

- Map intent labels `E/I/A/O` → integers `0/1/2/3`
- Sort by `conversationId` and `chatTime` (kept for parity with the context notebooks; doesn't affect k=0)
- Load DeBERTa-v3-base tokenizer

In [ ]:
LABEL2ID = {'E': 0, 'I': 1, 'A': 2, 'O': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

train_df['label'] = train_df['intentClass'].map(LABEL2ID)
valid_df['label'] = valid_df['intentClass'].map(LABEL2ID)

# Sort so context lookup is correct (no-op effect for k=0 but kept for parity)
for df in (train_df, valid_df):
    df.sort_values(['conversationId', 'chatTime'], inplace=True, kind='mergesort')
    df.reset_index(drop=True, inplace=True)

print('Label distribution (train):')
print(train_df['label'].value_counts().sort_index().rename(index=ID2LABEL))

Label distribution (train):
label
E     3528
I     1692
A     1719
O    19982
Name: count, dtype: int64


In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sanity check
test = 'ez mid report him'
print(f'Tokenization of {test!r}:')
print(' ', tokenizer.tokenize(test))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenization of 'ez mid report him':
  ['▁ez', '▁mid', '▁report', '▁him']


## 4. Dataset (k=0)

**k=0**: just the target message, no context. Identical wrapper to the unweighted baseline so the comparison is clean.

In [ ]:
MAX_K = 10  # kept for parity with the context notebooks

def build_context_index(df, max_k=10):
    ctx = [[] for _ in range(len(df))]
    for _, group in df.groupby('conversationId', sort=False):
        idxs = group.index.tolist()
        for pos, i in enumerate(idxs):
            ctx[i] = idxs[max(0, pos - max_k):pos]
    return ctx

train_context_idx = build_context_index(train_df, max_k=MAX_K)
valid_context_idx = build_context_index(valid_df, max_k=MAX_K)

In [ ]:
SEP = ' [SEP] '

class CONDAContextDataset(Dataset):
    """k=0 path: target utterance alone, matching conda_k0.ipynb exactly."""
    def __init__(self, df, context_idx, tokenizer, k, max_length=256):
        self.df = df
        self.context_idx = context_idx
        self.tokenizer = tokenizer
        self.k = k
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def _fmt(self, row, include_speaker):
        text = str(row['utterance'])
        return f"P{int(row['playerSlot'])}: {text}" if include_speaker else text

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if self.k == 0:
            input_str = self._fmt(row, include_speaker=False)
        else:
            ctx_rows = self.context_idx[idx][-self.k:]
            if ctx_rows:
                parts = [self._fmt(self.df.iloc[j], True) for j in ctx_rows]
                parts.append(self._fmt(row, True))
                input_str = SEP.join(parts)
            else:
                input_str = self._fmt(row, True)
        enc = self.tokenizer(
            input_str, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(int(row['label']), dtype=torch.long),
        }

MAX_LENGTH = 128 if K == 0 else 256
train_dataset = CONDAContextDataset(train_df, train_context_idx, tokenizer, K, MAX_LENGTH)
valid_dataset = CONDAContextDataset(valid_df, valid_context_idx, tokenizer, K, MAX_LENGTH)

sample = train_dataset[0]
decoded = tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f'Sample input (k={K}, label={ID2LABEL[sample["labels"].item()]}):')
print(decoded[:200])

Sample input (k=0, label=I):
ez 500[PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD


## 5. Class weights

We use **sqrt-inverse-frequency** weighting as the default. Pure inverse-frequency gives a 12× weight ratio (O≈0.34, I≈4.0) which tends to over-correct and tank precision on minority classes; sqrt-inverse is a gentler middle ground that is usually a better starting point on transformer fine-tunes.

The cell below computes both schemes and prints them so you can pick. Set `WEIGHT_SCHEME` to `'sqrt_inv'` (recommended), `'inv'`, or `'effective_num'` (Cui et al. 2019, β=0.999).

In [ ]:
WEIGHT_SCHEME = 'sqrt_inv'  # 'sqrt_inv' | 'inv' | 'effective_num'

counts = train_df['label'].value_counts().sort_index().values.astype(np.float64)  # [E, I, A, O]
total = counts.sum()
n_classes = len(counts)

w_inv = total / (n_classes * counts)
w_sqrt = np.sqrt(w_inv)
beta = 0.999
effective_num = 1.0 - np.power(beta, counts)
w_eff = (1.0 - beta) / effective_num
w_eff = w_eff / w_eff.sum() * n_classes  # normalize so mean weight = 1

schemes = {'inv': w_inv, 'sqrt_inv': w_sqrt, 'effective_num': w_eff}
print(f'{"class":<6}{"count":>8}{"inv":>10}{"sqrt_inv":>12}{"eff_num":>12}')
for i, c in enumerate(counts):
    name = ID2LABEL[i]
    print(f'{name:<6}{int(c):>8}{w_inv[i]:>10.3f}{w_sqrt[i]:>12.3f}{w_eff[i]:>12.3f}')

class_weights = torch.tensor(schemes[WEIGHT_SCHEME], dtype=torch.float32).to(device)
print(f'\nUsing scheme: {WEIGHT_SCHEME!r}')
print(f'class_weights (E,I,A,O): {class_weights.cpu().numpy().round(3).tolist()}')

class    count       inv    sqrt_inv     eff_num
E         3528     1.908       1.381       0.921
I         1692     3.978       1.994       1.096
A         1719     3.915       1.979       1.089
O        19982     0.337       0.580       0.894

Using scheme: 'sqrt_inv'
class_weights (E,I,A,O): [1.38100004196167, 1.99399995803833, 1.9789999723434448, 0.5799999833106995]


## 6. Load DeBERTa-v3-base + define metrics

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=torch.float32,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded ({n_params:,} parameters)')

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Model loaded (184,425,220 parameters)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per = f1_score(labels, preds, average=None, zero_division=0, labels=[0,1,2,3])
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_E': f1_per[0],
        'f1_I': f1_per[1],
        'f1_A': f1_per[2],
        'f1_O': f1_per[3],
    }

## 7. Weighted Trainer

Subclass `Trainer` and override `compute_loss` to use class-weighted CE. Everything else (optimizer, scheduler, eval loop, checkpointing) is unchanged.

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 8. Train

Same hyperparameters as `conda_k0.ipynb` so the only difference is the loss.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights,
)

ckpts = [p for p in CKPT_DIR.glob('checkpoint-*') if p.is_dir()]
resume = bool(ckpts)
if resume:
    print(f'Resuming from existing checkpoint(s): {[p.name for p in ckpts]}')

t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=resume)
elapsed_min = (time.time() - t0) / 60
print(f'\nTraining done in {elapsed_min:.1f} min')
print(f'Final train loss: {train_result.training_loss:.4f}')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 E,F1 I,F1 A,F1 O
1,0.784287,0.689195,0.876978,0.741896,0.776417,0.653552,0.607018,0.930596
2,0.617232,0.571259,0.903499,0.805128,0.810104,0.708808,0.757548,0.944050
3,0.422788,0.543361,0.910742,0.825887,0.839071,0.743083,0.773698,0.947697
4,0.371898,0.574059,0.904168,0.816177,0.827303,0.726054,0.767123,0.944227
5,0.324687,0.560620,0.904948,0.819755,0.831497,0.728111,0.775087,0.944327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye


Training done in 19.0 min
Final train loss: 0.5898


## 9. Final evaluation on the validation set

In [ ]:
eval_metrics = trainer.evaluate()
print('Validation metrics (best model):')
for kk, v in eval_metrics.items():
    if isinstance(v, float):
        print(f'  {kk:30s} {v:.4f}')

Validation metrics (best model):
  eval_loss                      0.5437
  eval_accuracy                  0.9107
  eval_f1_macro                  0.8259
  eval_f1_E                      0.8391
  eval_f1_I                      0.7431
  eval_f1_A                      0.7737
  eval_f1_O                      0.9477
  eval_runtime                   21.8697
  eval_samples_per_second        410.3390
  eval_steps_per_second          12.8490
  epoch                          5.0000


In [ ]:
preds_output = trainer.predict(valid_dataset)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids
print(classification_report(
    y_true, y_pred,
    target_names=['E', 'I', 'A', 'O'],
    digits=4, zero_division=0,
))

              precision    recall  f1-score   support

           E     0.8545    0.8242    0.8391      1183
           I     0.8744    0.6460    0.7431       582
           A     0.7665    0.7810    0.7737       580
           O     0.9350    0.9608    0.9477      6629

    accuracy                         0.9107      8974
   macro avg     0.8576    0.8030    0.8259      8974
weighted avg     0.9095    0.9107    0.9089      8974



## 10. Compare against the unweighted k=0 baseline

Reminder of the unweighted k=0 numbers (from `conda_k0.ipynb`):

| class | precision | recall | f1   |
|-------|-----------|--------|------|
| E     | 0.7949    | 0.8453 | 0.8193 |
| I     | 0.9245    | 0.6100 | 0.7350 |
| A     | 0.8124    | 0.7466 | 0.7781 |
| O     | 0.9359    | 0.9599 | 0.9477 |
| **macro** | **0.8669** | **0.7904** | **0.8200** |

What to look for in the weighted run above:
- **Class I recall** should go up (from 0.61). If it climbs to ~0.75–0.80 without crushing precision, weighting is helping.
- **Class O recall** will likely drop slightly. If it falls below ~0.92 the weighting is too aggressive — try `WEIGHT_SCHEME='effective_num'` or lower the `inv` weights.
- **Macro-F1**: if it’s within ±0.005 of 0.8200, the schemes are effectively a wash and the report can justify keeping unweighted for the main ablation. If weighted is clearly higher, consider rerunning the full k-sweep weighted.

## 11. Save model + metadata to Drive

In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

meta = {
    'k': K,
    'weighted': True,
    'weight_scheme': WEIGHT_SCHEME,
    'class_weights': class_weights.cpu().numpy().round(6).tolist(),
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'train_minutes': elapsed_min,
    'final_train_loss': float(train_result.training_loss),
    'eval_metrics': {kk: float(v) for kk, v in eval_metrics.items() if isinstance(v, (int, float))},
    'label2id': LABEL2ID,
    'id2label': ID2LABEL,
    'sep_token': SEP.strip(),
    'speaker_tag_format': 'P{playerSlot}:',
}
with open(MODEL_DIR / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved model to {MODEL_DIR}')
for p in sorted(MODEL_DIR.iterdir()):
    size_mb = p.stat().st_size / 1e6
    print(f'  {p.name:30s} {size_mb:8.2f} MB')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to /content/conda_ksweep/k0_weighted_model
  config.json                        0.00 MB
  meta.json                          0.00 MB
  model.safetensors                737.73 MB
  tokenizer.json                     8.34 MB
  tokenizer_config.json              0.00 MB
  training_args.bin                  0.01 MB


In [ ]:
# Optional: delete intermediate checkpoints to free Drive space
import shutil
for ckpt in CKPT_DIR.glob('checkpoint-*'):
    if ckpt.is_dir():
        shutil.rmtree(ckpt)
        print(f'Removed {ckpt}')

Removed /content/conda_ksweep/k0_weighted_checkpoints/checkpoint-5049
Removed /content/conda_ksweep/k0_weighted_checkpoints/checkpoint-8415


## 12. (Optional) Download the model archive

Mirrors the final cell of `conda_k0.ipynb` so the demo website can pick this model up the same way.

In [ ]:
import tarfile, os
from google.colab import files

LOCAL_MODEL_DIR = f'/content/k{K}_weighted_model'
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

trainer.save_model(LOCAL_MODEL_DIR)
tokenizer.save_pretrained(LOCAL_MODEL_DIR)
with open(f'{LOCAL_MODEL_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

archive_path = f'/content/k{K}_weighted_model.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(LOCAL_MODEL_DIR, arcname=f'k{K}_weighted_model')

print(f'Archive: {archive_path} ({os.path.getsize(archive_path)/1e6:.1f} MB)')
files.download(archive_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Archive: /content/k0_weighted_model.tar.gz (586.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>